# Lab 9: 0/1 Knapsack using Branch-and-Bound

## Objective
Implement 0/1 Knapsack using the Least-Cost Branch-and-Bound (LCBB) method with priority queues and an upper-bound calculation to prune non-promising search branches.


In [1]:
import heapq
from dataclasses import dataclass, field
from typing import List, Tuple

@dataclass(order=True)
class PriorityNode:
    priority: float  # -upper_bound for max-heap via min-heap
    level: int = field(compare=False)
    profit: int = field(compare=False)
    weight: int = field(compare=False)
    bound: float = field(compare=False)

def calculate_bound(level: int, weight: int, profit: int, capacity: int, n: int, items: List[Tuple[int, int]]) -> float:
    if weight >= capacity:
        return 0.0
    bound = float(profit)
    total_w = weight
    
    for i in range(level, n):
        w, p = items[i]
        if total_w + w <= capacity:
            total_w += w
            bound += p
        else:
            # Add fractional part of remaining item (greedy upper bound)
            bound += (capacity - total_w) * (p / w)
            break
            
    return bound

def knapsack_branch_and_bound(weights: List[int], values: List[int], capacity: int) -> int:
    n = len(weights)
    # Sort items based on value-to-weight ratio descending
    items = sorted(zip(weights, values), key=lambda x: x[1] / x[0], reverse=True)
    
    pq = []
    initial_bound = calculate_bound(0, 0, 0, capacity, n, items)
    heapq.heappush(pq, PriorityNode(priority=-initial_bound, level=0, profit=0, weight=0, bound=initial_bound))
    
    max_profit = 0
    nodes_explored = 0
    
    while pq:
        node = heapq.heappop(pq)
        nodes_explored += 1
        
        if node.bound <= max_profit:
            continue
            
        if node.level == n:
            continue
            
        w, p = items[node.level]
        
        # Branch 1: Include item[level]
        next_w = node.weight + w
        next_p = node.profit + p
        if next_w <= capacity:
            if next_p > max_profit:
                max_profit = next_p
            next_bound = calculate_bound(node.level + 1, next_w, next_p, capacity, n, items)
            if next_bound > max_profit:
                heapq.heappush(pq, PriorityNode(priority=-next_bound, level=node.level + 1, profit=next_p, weight=next_w, bound=next_bound))
                
        # Branch 2: Exclude item[level]
        next_bound_exclude = calculate_bound(node.level + 1, node.weight, node.profit, capacity, n, items)
        if next_bound_exclude > max_profit:
            heapq.heappush(pq, PriorityNode(priority=-next_bound_exclude, level=node.level + 1, profit=node.profit, weight=node.weight, bound=next_bound_exclude))
            
    print(f"Nodes Explored: {nodes_explored}")
    return max_profit

weights = [2, 3, 4, 5]
values = [3, 4, 5, 6]
capacity = 5

max_val = knapsack_branch_and_bound(weights, values, capacity)
print(f"Knapsack Capacity: {capacity}")
print(f"Weights: {weights}, Values: {values}")
print(f"Optimal Value (Branch & Bound): {max_val}")


Nodes Explored: 7
Knapsack Capacity: 5
Weights: [2, 3, 4, 5], Values: [3, 4, 5, 6]
Optimal Value (Branch & Bound): 7
